# Visual Example: Input-Based vs Curvature-Based Grid Adaptation


In [ ]:
from jaxkan.models.KAN import KAN
from jaxkan.grids import (
    UniformDensity,
    CurvatureDensity,
    MixedAdaptation
)

import jax
import jax.numpy as jnp
from flax import nnx
import optax

import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## Configuration

In [ ]:
# Training configuration
NUM_EPOCHS = 1000
LEARNING_RATE = 0.005
SEED = 42

# Function parameters
PEAK_CENTER = 0.0
PEAK_WIDTH = 0.05
INPUT_RANGE = [-1.0, 1.0]

# Data generation
N_TRAIN = 1000

# Architecture
ARCHITECTURE = [1, 10, 1]

# Grid configuration
G_INITIAL = 3
G_FINAL = 10

## Define Target Function

In [ ]:
def sharp_gaussian(x, center=PEAK_CENTER, width=PEAK_WIDTH):
    
    if x.ndim == 1:
        x = x.reshape(-1, 1)
    return jnp.exp(-((x[:, 0] - center) ** 2) / (2 * width ** 2))

x_test = jnp.linspace(-1, 1, 100).reshape(-1, 1)
y_test = sharp_gaussian(x_test)

## Generate Training Data

In [ ]:
def generate_uniform_data(n_samples, seed):
    
    key = jax.random.key(seed)
    x = jax.random.uniform(key, shape=(n_samples, 1), 
                          minval=INPUT_RANGE[0], 
                          maxval=INPUT_RANGE[1])
    y = sharp_gaussian(x)
    return x, y

X, y = generate_uniform_data(N_TRAIN, SEED)

## Training Functions

In [ ]:
@nnx.jit
def train_step(model, optimizer, X_batch, y_batch):
    def loss_fn(model):
        residual = model(X_batch) - y_batch.reshape(-1, 1)
        loss = jnp.mean(residual**2)
        return loss
    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    
    return loss

@nnx.jit
def compute_curvature(model, X):
    
    epsilon = 1e-3
    n_in = X.shape[1]
    curvatures = jnp.zeros(X.shape[0])
    
    for dim in range(n_in):
        h = jnp.zeros((1, n_in))
        h = h.at[0, dim].set(epsilon)
        
        X_plus = X + h
        X_minus = X - h
        
        f_center = model(X)
        f_plus = model(X_plus)
        f_minus = model(X_minus)
        
        second_deriv = (f_plus - 2 * f_center + f_minus) / (epsilon ** 2)
        curvatures = curvatures + jnp.sum(jnp.abs(second_deriv), axis=1)
    
    return curvatures

## Training

In [ ]:
# Create model for uniform data with input-based adaptation
req_params = {'k': 3, 'G': G_INITIAL, 'init_scheme': {'type': 'glorot_fine'}}
model_input = KAN(
    layer_dims=ARCHITECTURE,
    layer_type='spline',
    required_parameters=req_params,
    seed=SEED
)

# Create optimizer
optimizer = nnx.Optimizer(model_input, optax.adam(LEARNING_RATE), wrt=nnx.Param)

# Setup adaptation
idf_input = UniformDensity()
strategy_input = MixedAdaptation(grid_e=0.0)

# Train for NUM_EPOCHS with grid adaptation at final epoch
print(f"Training model (input-based) for {NUM_EPOCHS} epochs...")
for epoch in range(NUM_EPOCHS):
    
    loss = train_step(model_input, optimizer, X, y)
    if (epoch + 1) % 200 == 0:
        print(f"  Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {loss:.6f}")
        
    # Adapt grid at final epoch
    if epoch == NUM_EPOCHS - 1:
        print(f"  Epoch {epoch+1}: Adapting grid from G={G_INITIAL} to G={G_FINAL}")
        model_input.update_grids(
            X,
            idf=idf_input,
            strategy=strategy_input,
            grid_size_new=G_FINAL
        )
        # Reset optimizer after grid change
        optimizer = nnx.Optimizer(model_input, optax.adam(LEARNING_RATE), wrt=nnx.Param)

# Extract knots after adaptation
knots_input = model_input.layers[0].grid.knots

In [ ]:
# Create model for uniform data with curvature-based adaptation
model_curv = KAN(
    layer_dims=ARCHITECTURE,
    layer_type='spline',
    required_parameters=req_params,
    seed=SEED
)

# Create optimizer
optimizer = nnx.Optimizer(model_curv, optax.adam(LEARNING_RATE), wrt=nnx.Param)

# Setup adaptation
idf_curv = CurvatureDensity()
strategy_curv = MixedAdaptation(grid_e=0.0)

# Train for NUM_EPOCHS with grid adaptation at final epoch
print(f"\nTraining model (curvature-based) for {NUM_EPOCHS} epochs...")
for epoch in range(NUM_EPOCHS):
    
    loss = train_step(model_curv, optimizer, X, y)
    if (epoch + 1) % 200 == 0:
        print(f"  Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {loss:.6f}")
        
    # Adapt grid at final epoch
    if epoch == NUM_EPOCHS - 1:
        print(f"  Epoch {epoch+1}: Adapting grid from G={G_INITIAL} to G={G_FINAL}")
        curvatures = compute_curvature(model_curv, X)
        model_curv.update_grids(
            X,
            idf=idf_curv,
            strategy=strategy_curv,
            grid_size_new=G_FINAL,
            curvatures=curvatures
        )
        # Reset optimizer after grid change
        optimizer = nnx.Optimizer(model_curv, optax.adam(LEARNING_RATE), wrt=nnx.Param)

# Extract knots after adaptation
knots_curv = model_curv.layers[0].grid.knots

## Save Data for Plotting

In [ ]:
# Create reference function data for plotting
x_ref = jnp.linspace(INPUT_RANGE[0], INPUT_RANGE[1], 1000).reshape(-1, 1)
y_ref = sharp_gaussian(x_ref)

# Get model predictions on reference grid
y_input = model_input(x_ref)
y_curv = model_curv(x_ref)

# Package all data
plot_data = {
    'config': {
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'seed': SEED,
        'peak_center': PEAK_CENTER,
        'peak_width': PEAK_WIDTH,
        'input_range': INPUT_RANGE
    },
    'reference_function': {
        'x': np.array(x_ref),
        'y': np.array(y_ref)
    },
    'model': {
        'training_data': {
            'x': np.array(X),
            'y': np.array(y)
        },
        'knots_input': np.array(knots_input),
        'knots_curvature': np.array(knots_curv),
        'model_input': np.array(y_input),
        'model_curvature': np.array(y_curv)
    }

}

# Save to file
output_path = Path('results/example_plot_data.pkl')

with open(output_path, 'wb') as f:
    pickle.dump(plot_data, f)

print(f"\nData saved to: {output_path}")

## Visualization Function

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def plot_adaptation_comparison(data, save_path=None, show_target=True, show_input_model=True, show_curv_model=True,
                               figsize=(10, 8), title_size=16, label_size=14, legend_size=11, tick_size=11,
                               data_color='#7f7f7f',
                               ref_color='black',
                               input_color='#d62728',
                               curv_color='#1f77b4'):
    
    plt.rcParams['font.family'] = 'sans-serif'

    stick_height = 0.3
    
    fig = plt.figure(figsize=figsize)
    
    gs = gridspec.GridSpec(3, 1, height_ratios=[3, 1, 1], hspace=0.15)
    
    x_min, x_max = data['config']['input_range']
    x_pad = (x_max - x_min) * 0.05
    x_limits = (x_min - x_pad, x_max + x_pad)
    
    # Extract data
    x_ref = data['reference_function']['x'].flatten()
    y_ref = data['reference_function']['y'].flatten()
    
    # ==========================================
    # Row 1: Main Function Approximation
    # ==========================================
    ax1 = fig.add_subplot(gs[0, 0])
    
    # 1. Training Data 
    ax1.scatter(data['model']['training_data']['x'], 
                data['model']['training_data']['y'], 
                s=15, alpha=0.1, color=data_color, label='Training Samples', zorder=1)

    # 2. Reference Function
    if show_target:
        ax1.plot(x_ref, y_ref, '-', linewidth=2, color=ref_color, 
                 label='Ground Truth', zorder=2, alpha=0.9)
    
    # 3. Models
    if show_input_model and 'model_input' in data['model']:
        ax1.plot(x_ref, data['model']['model_input'].flatten(), 
                '--', linewidth=2, alpha=0.9, color=input_color, 
                label='Input-Based', zorder=3)
        
    if show_curv_model and 'model_curvature' in data['model']:
        ax1.plot(x_ref, data['model']['model_curvature'].flatten(), 
                '--', linewidth=2, alpha=0.9, color=curv_color, 
                label='Curvature-Based', zorder=4)
    
    ax1.set_ylabel(r'Model Output', fontsize=label_size, labelpad=10)
    ax1.legend(loc='upper right', frameon=True, fontsize=legend_size, framealpha=0.95)
    ax1.grid(True, linestyle=':', alpha=0.4)
    ax1.set_xlim(x_limits)
    ax1.tick_params(axis='x', labelbottom=False)
    
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # ==========================================
    # Helper for Knot Rows
    # ==========================================
    def plot_knot_row(ax, knots, color, label):
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        
        ax.axhline(y=0, color='gray', linewidth=1, alpha=0.5)
        
        # Draw Knots as "Stems" (Lines + Dots)
        ax.vlines(knots, ymin=0, ymax=stick_height, color=color, linewidth=2, alpha=0.8)
        ax.scatter(knots, np.full_like(knots, stick_height/2), 
                   marker='o', s=31, facecolors='white', edgecolors=color, linewidth=1.5, 
                   label=label, zorder=3)
        
        # Set limits
        ax.set_ylim(0, 0.8)
        ax.set_yticks([])
        ax.set_xlim(x_limits)
        
        # Legend
        ax.legend(loc='upper right', frameon=True, fontsize=legend_size, 
          framealpha=0.95, edgecolor='#dddddd')
        
        # Vertical grid only
        ax.grid(True, axis='x', linestyle=':', alpha=0.4)

    # ==========================================
    # Row 2: Input-Based Knots
    # ==========================================
    ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
    plot_knot_row(ax2, data['model']['knots_input'], input_color, 'Input-Based Knots')
    ax2.tick_params(axis='x', labelbottom=False)
    ax2.spines['bottom'].set_visible(False) 

    # ==========================================
    # Row 3: Curvature-Based Knots
    # ==========================================
    ax3 = fig.add_subplot(gs[2, 0], sharex=ax1)
    plot_knot_row(ax3, data['model']['knots_curvature'], curv_color, 'Curvature-Based Knots')
    
    # Bottom row gets the labels
    ax3.set_xlabel(r'Model Input', fontsize=label_size, labelpad=15)
    ax3.tick_params(axis='x', labelsize=tick_size)
    ax3.spines['bottom'].set_visible(False)

    # Align y-labels
    fig.align_ylabels([ax1, ax2, ax3])
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved to: {save_path}")
    
    plt.show()

## Generate Plot

In [ ]:
# Load data and create plot
with open('results/example_plot_data.pkl', 'rb') as f:
    plot_data = pickle.load(f)

# Create and save the plot
plot_adaptation_comparison(plot_data, save_path='results/illustrative_comparison.pdf', show_target=False, show_input_model=True, 
                           show_curv_model=True, figsize=(8, 6), label_size=12, legend_size=10, tick_size=10)